# RAG System with RAGBench Dataset

Building a Retrieval-Augmented Generation system using RAGBench with BAAI/bge-large-en-v1.5 embeddings and FAISS vector store.

## 1. Install Dependencies

In [102]:
!pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [103]:
import numpy as np
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import json
from typing import List, Tuple

print("Dependencies loaded!")

Dependencies loaded!


## 3.1 Loading environment variables & defining models

In [104]:
# Step - 4: Set up Groq client

import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env file
load_dotenv(override=True)

# Get API keys & models from environment
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"HuggingFace token loaded: {bool(HF_TOKEN)}")
print(f"Groq API key loaded: {bool(GROQ_API_KEY)}")

EMBEDDING_MODEL="pritamdeka/S-PubMedBert-MS-MARCO"
RERANKING_MODEL="ncbi/MedCPT-Cross-Encoder"
GENERATION_MODEL="llama-3.1-8b-instant"
JUDGING_MODEL="llama-3.3-70b-versatile"

CHUNK_SIZE = 80

HuggingFace token loaded: True
Groq API key loaded: True


## 4. Load RAGBench Covidqa Dataset

In [105]:
# Load RAGBench dataset
dataset = load_dataset("galileo-ai/ragbench", name="covidqa", split="test")
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nFirst sample:")
for key, value in dataset[0].items():
    if isinstance(value, str) and len(value) > 100:
        print(f"{key}: {value[:150]}...")
    else:
        print(f"{key}: {value}")

Dataset loaded: 246 samples
Columns: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score']

First sample:
id: 1421
question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
documents: ['Title: Type I Interferon Receptor Deficiency in Dendritic Cells Facilitates Systemic Murine Norovirus Persistence Despite Enhanced Adaptive Immunity\nPassage: successful treatment for HCV serves to circumvent the viral inhibit

## 5. Initialzie Embedding Model & Reranking Model objects

In [106]:
# Load BAAI/bge-large-en-v1.5 embedding model
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding Model loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Load BAAI/bge-reranker-v2-m3"= reranking model
print(f"Loading embedding model: {RERANKING_MODEL}")
rerank_model = CrossEncoder(RERANKING_MODEL)
print(f"Reranking Model loaded")

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57396.95it/s]
/var/folders/fr/r16pl_8n3xj0lfytz1g6qsc40000gn/T/ipykernel_35262/2051931484.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding Model loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


Embedding Model loaded. Embedding dimension: 768
Loading embedding model: ncbi/MedCPT-Cross-Encoder


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 51816.54it/s]


Reranking Model loaded


## 6. Utils for data chunking

In [107]:
## Parsing title and passage from row document
import re
from typing import List


def parse_document(raw_doc: str):

    title_match = re.search(
        r"Title:\s*(.*?)\n",
        raw_doc,
        re.DOTALL
    )

    passage_match = re.search(
        r"Passage:\s*(.*)",
        raw_doc,
        re.DOTALL
    )

    title = (
        title_match.group(1).strip()
        if title_match else ""
    )

    passage = (
        passage_match.group(1).strip()
        if passage_match else raw_doc.strip()
    )

    return title, passage

## Split passage into multiple strings
def split_into_sentences(text: str) -> List[str]:

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text.strip()
    )

    return [s.strip() for s in sentences if s.strip()]

## Format document ( not using currently )
def get_formatted_content(title, passage):
    return f"""
[TITLE]
{title}

[CONTENT]
{passage}
""".strip()


## chunk single passage string
def chunk_passage_data(
    passage_data: str,
    max_words: int = CHUNK_SIZE,
    overlap_sentences: int = 1
):

    sentences = split_into_sentences(passage_data)

    chunks = []

    current_chunk = []
    current_words = 0

    i = 0

    while i < len(sentences):

        sentence = sentences[i]
        sentence_words = len(sentence.split())

        # -------------------------------------------------
        # Handle extremely large single sentence
        # -------------------------------------------------

        if sentence_words > max_words:

            # flush existing chunk first
            if current_chunk:

                final_chunk = " ".join(current_chunk)

                chunks.append(final_chunk)

                current_chunk = []
                current_words = 0

            # store oversized sentence alone
            final_chunk = sentence

            chunks.append(final_chunk)

            i += 1
            continue

        # -------------------------------------------------
        # Normal chunk overflow handling
        # -------------------------------------------------

        if current_words + sentence_words > max_words:

            if current_chunk:
                chunks.append(" ".join(current_chunk))

            overlap = []

            if overlap_sentences > 0:
                overlap = current_chunk[-overlap_sentences:]

            current_chunk = overlap
            current_words = sum(len(s.split()) for s in current_chunk)

            # safety fallback
            if current_words + sentence_words > max_words:
                current_chunk = []
                current_words = 0

            continue

        # -------------------------------------------------
        # Add sentence normally
        # -------------------------------------------------

        current_chunk.append(sentence)

        current_words += sentence_words

        i += 1

    # -------------------------------------------------
    # Final chunk
    # -------------------------------------------------

    if current_chunk:

        final_chunk = " ".join(current_chunk)

        chunks.append(final_chunk)

    return chunks

## 7. Creating passage chunks with meatadata

In [108]:
chunked_data = []
# ---------------------------------------------------------
# Prepare structured documents for embeddings
# ---------------------------------------------------------

for sample_idx, sample in enumerate(dataset):

    raw_docs = [
        doc.strip()
        for doc in sample.get("documents", [])
        if doc and doc.strip()
    ]

    # running nested loops instead of extend so I can perform operations on individual string

    for doc_idx, raw_doc in enumerate(raw_docs):

        title, passage = parse_document(raw_doc=raw_doc)

        chunks = chunk_passage_data(passage)

        for chunk_idx, chunk_text in enumerate(chunks):

            chunked_data.append({
                "text": chunk_text,
                "metadata": {
                    "title": title,
                    "doc_id": f"sample_{sample_idx}_doc_{doc_idx}",
                    "chunk_id": f"sample_{sample_idx}_doc_{doc_idx}_chunk_{chunk_idx}",
                    "sample_index": sample_idx
                }
            })

# ---------------------------------------------------------
# Debug / Inspection
# ---------------------------------------------------------

print(f"Extracted {len(chunked_data)} chunked data\n")

print("Sample formatted documents:\n")

for idx, current_document in enumerate(chunked_data[:10]):

    print(f"--- Document {idx+1} ---")
    print(current_document)
    print("\n" + "=" * CHUNK_SIZE + "\n")

Extracted 1544 chunked data

Sample formatted documents:

--- Document 1 ---
{'text': 'successful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV may be an example of a medically relevant persistent viral infection that persists due, in part, to loss of innate immune function. Persistence of other continuously replicating RNA viruses, such as chikungunya, measles, polyomavirus, may be similarly due to ineffective innate responses.', 'metadata': {'title': 'Type I Interferon Receptor Deficiency in Dendritic Cells Facilitates Systemic Murine Norovirus Persistence Despite Enhanced Adaptive Immunity', 'doc_id': 'sample_0_doc_0', 'chunk_id': 'sample_0_doc_0_chunk_0', 'sample_index': 0}}


--- Document 2 ---
{'text': 'Results suggest that HAstV infection is not able to disrupt the innate immune sensing pathway induced by polyI:C . Only a previous infection with RV was able to reduce by 60% the IFN-β mRNA levels produced after polyI:C transfection, altho

## 8. Generate Embeddings

In [109]:
texts = [
    chunk["text"]
    for chunk in chunked_data
]

print(f"Generating embeddings for {len(texts)} data chunks...")

embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Data type: {embeddings.dtype}")

Generating embeddings for 1544 data chunks...


Batches: 100%|██████████| 49/49 [00:15<00:00,  3.24it/s]

Embeddings shape: (1544, 768)
Embedding dimension: 768
Data type: float32


## 9. Create FAISS Index

In [110]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

## 10. Retrieval Function

In [111]:
from typing import List, Dict

def retrieve(
    query: str,
    top_k: int = 5,
    initial_k: int = 20
) -> List[Dict]:

    # -----------------------------------------------------
    # Step 1: Encode query
    # -----------------------------------------------------

    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # -----------------------------------------------------
    # Step 2: Initial FAISS retrieval
    # -----------------------------------------------------

    distances, indices = index.search(
        query_embedding,
        initial_k
    )

    candidate_docs = []

    for idx, distance in zip(indices[0], distances[0]):

        if idx >= 0:

            candidate_docs.append({
                "text": chunked_data[idx]["text"],
                "metadata": chunked_data[idx]["metadata"],
                "faiss_score": float(distance)
            })

    # -----------------------------------------------------
    # Step 3: Prepare reranker pairs
    # -----------------------------------------------------

    rerank_pairs = [
        [query, doc["text"]]
        for doc in candidate_docs
    ]

    # -----------------------------------------------------
    # Step 4: Cross-encoder reranking
    # -----------------------------------------------------

    rerank_scores = rerank_model.predict(
        rerank_pairs
    )

    # -----------------------------------------------------
    # Step 5: Attach rerank scores
    # -----------------------------------------------------

    for doc, score in zip(
        candidate_docs,
        rerank_scores
    ):

        doc["rerank_score"] = float(score)

    # -----------------------------------------------------
    # Step 6: Sort by rerank score
    # -----------------------------------------------------

    reranked_results = sorted(
        candidate_docs,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # -----------------------------------------------------
    # Step 7: Return top-k
    # -----------------------------------------------------

    return reranked_results[:top_k]

# --------------------------
# Testing retrieval
# --------------------------

# Test query
test_query = "What is the main topic?"

results = retrieve(test_query, top_k=3)

print(f"Query: {test_query}")

print("\nTop-3 retrieved documents:")

for i, result in enumerate(results):

    title = result["metadata"]["title"]

    text = result["text"]

    faiss_score = result["faiss_score"]

    rerank_score = result["rerank_score"]

    print(f"\n{i+1}.")
    print(f"Title: {title}")

    print(f"FAISS Score: {faiss_score:.4f}")
    print(f"Rerank Score: {rerank_score:.4f}")

    print(f"\n{text[:300]}...")

    print("\n" + "=" * 80)

Query: What is the main topic?

Top-3 retrieved documents:

1.
Title: Missing and accounted for: gaps and areas of wealth in the public health review literature
FAISS Score: 0.8710
Rerank Score: 0.9782

Three categories were used to define availability of reviews within each topic area: few, representing 1-150 reviews; moderate, representing 151-300 reviews; and, many, representing topic areas possessing greater than 301 reviews. Reviews that addressed multiple topics were accounted for within each...


2.
Title: Viruses and Evolution – Viruses First? A Personal Perspective
FAISS Score: 0.8773
Rerank Score: 0.8930

Abstract: The discovery of exoplanets within putative habitable zones revolutionized astrobiology in recent years. It stimulated interest in the question about the origin of life and its evolution. Here, we discuss what the roles of viruses might have been at the beginning of life and during evoluti...


3.
Title: Missing and accounted for: gaps and areas of wealth in the pu

## 11. Load Groq client

In [112]:
# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)
print("Clients initialized!")

test_response = client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": "Say 'Groq is connected' in exactly 3 words."}],
    max_tokens=20
)

print(test_response.choices[0].message.content)


Clients initialized!
Groq is connected.


## 12. Generate response

In [113]:
def generate_response(query: str, retrieved_docs: List, model=GENERATION_MODEL) -> str:
    """Use retrieved_docs and generate response using Groq."""
    
    context = "\n\n".join([
    f"""
[Document {i+1}]
Title: {doc['metadata']['title']}
Content:
{doc['text']}
""".strip()
    for i, doc in enumerate(retrieved_docs)
])
    
    # Create prompt for Groq
    prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {query}

Answer:"""
    
    # Call Groq API
    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return response.choices[0].message.content


## 13. Test response generation

In [114]:
# Test with a query
test_query = "What is the main topic?"
retrieved_docs = retrieve(test_query)
response = generate_response(test_query, retrieved_docs)

print(f"Query: {test_query}\n")
print(f"Generated Response:\n{response}")

Query: What is the main topic?

Generated Response:
Based on the provided context, it appears that the main topic is related to public health review literature, specifically gaps and areas of wealth in the public health review literature.


## 14. Split into sentences Util

In [115]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

def split_into_sentences(text):
    """Split a text into clean sentences using NLTK punkt tokenizer."""
    sentences = sent_tokenize(text.strip())
    # Remove empty or very short fragments (< 10 chars)
    return [s.strip() for s in sentences if len(s.strip()) >= 10]

## 15. Build doc sentences ( build retrieved document sentences )

In [116]:
def build_doc_sentences(retrieved_docs):
    """
    Convert retrieved documents into sentence-level structures
    for TRACe evaluation.

    Input:
        [
            {
                "text": ...,
                "metadata": ...,
                "faiss_score": ...,
                "rerank_score": ...
            },
            ...
        ]

    Output:
        [
            [
                ["d0s0", "Sentence 1"],
                ["d0s1", "Sentence 2"]
            ],
            ...
        ]
    """

    doc_sentences = []

    for doc_idx, doc in enumerate(retrieved_docs):

        text = doc["text"]

        sentences = split_into_sentences(text)

        doc_sent_pairs = []

        for sent_idx, sentence in enumerate(sentences):

            # Collision-safe sentence IDs
            key = f"d{doc_idx}s{sent_idx}"

            doc_sent_pairs.append([
                key,
                sentence
            ])

        doc_sentences.append(doc_sent_pairs)

    return doc_sentences

## 16. Build response sentences ( convert llm generated response into multiple sentence format.)

In [117]:
def build_response_sentences(response):
    """
    Convert response into sentence-key format.
    Response sentence 0 → key "a", sentence 1 → key "b" ...
    Returns: list of [key, sentence] pairs (mirrors dataset structure)
    """
    sentences = split_into_sentences(response)
    response_sentences = []
    for idx, sentence in enumerate(sentences):
        letter = chr(ord('a') + idx % 26)
        response_sentences.append([letter, sentence])
    return response_sentences

## 17. Test senetence formations for response and retrieved documents

In [118]:
example = dataset[0]
query = example['question']
retrieved_docs = retrieve(query, top_k=5)
our_response       = generate_response(query, retrieved_docs)
print(f"Query: {query} \n")
print(f"Retrived documents: {retrieved_docs} \n")
print(f"Generated response: {our_response} \n")
test_doc_sents = build_doc_sentences(retrieved_docs)
print(f"Doc sentence structure: {len(test_doc_sents)} docs")
for i, doc in enumerate(test_doc_sents):
    print(f"  Doc {i}: {len(doc)} sentences: \n")
    for j, (key, doc_sent) in enumerate(doc):
        print(f"{j}.  Doc Sent key {key}: Sentence: {doc_sent}")

test_resp = generate_response(query, retrieved_docs)
test_resp_sents = build_response_sentences(test_resp)
print(f"\nResponse sentences:")
for key, sent in test_resp_sents:
    print(f"  {key}. {sent}")

Query: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance? 

Retrived documents: [{'text': 'transcriptomic blueprints for this IFN response are expressed constitutively, even in the absence of stimulation by viral RNA or DNA . In non-flying mammals, constitutive IFN expression would likely elicit widespread inflammation and concomitant immunopathology upon viral infection, but bats support unique adaptations to combat inflammation that may have evolved to mitigate metabolic damage induced during flight . The extent to which constitutive IFN-a expression signifies constitutive antiviral defense in the form of functional IFN-a protein remains unresolved.', 'metadata': {'title': 'Accelerated viral dynamics in bat cell lines, with implications for zoonotic emergence', 'doc_id': 'sample_97_doc_0', 'chunk_id': 'sample_97_doc_0_chunk_0', 'sample_index': 97}, 'faiss_score': 0.9204219579696655, 'rerank_score': 0.9773678183555603}, {'text': 'Several

# Save FAISS index
faiss.write_index(index, "data/vectordb/faiss_index.bin")
print("FAISS index saved to data/vectordb/faiss_index.bin")

# Save documents and metadata
with open("data/cache/documents.json", "w") as f:
    json.dump({
        "documents": chunked_data,
        "metadata": {
            "dataset_name": "covidqa",
            "chunk_size": CHUNK_SIZE,
            "num_chunks": len(chunked_data)
        }
    }, f, indent=2)
print("Documents and metadata saved to data/cache/documents.json")

# Save retrieval results
df_results.to_csv(
    "data/reports/retrieval_results.csv",
    index=False
)
print("Retrieval results saved to data/reports/retrieval_results.csv")

In [ ]:
# To load the index later:
# index = faiss.read_index("data/vectordb/faiss_index.bin")
# with open("data/cache/documents.json", "r") as f:
#     data = json.load(f)
#     documents = data["documents"]

## 19. E2E pipeline

In [120]:
example = dataset[0]
query = example['question']
retrieved_docs = retrieve(query, top_k=5)
our_response       = generate_response(query, retrieved_docs)
print(f"Question: \n {query} \n")
print(f"Retrived documents: \n {retrieved_docs} \n")
print(f"Generated response: \n {our_response} \n")

scores             = judge_trace_scores(query, retrieved_docs, our_response)

print("=== END-TO-END PIPELINE TEST (example 0) ===")
print(f"\n--- TRACe Scores ---")
print(f"  relevance:    {scores['relevance_score']}   (GT: {example['relevance_score']:.4f})")
print(f"  utilization:  {scores['utilization_score']}   (GT: {example['utilization_score']:.4f})")
print(f"  completeness: {scores['completeness_score']}   (GT: {example['completeness_score']:.4f})")
print(f"  adherence:    {scores['adherence_score']}    (GT: {example['adherence_score']})")

Question: 
 Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance? 

Retrived documents: 
 [{'text': 'transcriptomic blueprints for this IFN response are expressed constitutively, even in the absence of stimulation by viral RNA or DNA . In non-flying mammals, constitutive IFN expression would likely elicit widespread inflammation and concomitant immunopathology upon viral infection, but bats support unique adaptations to combat inflammation that may have evolved to mitigate metabolic damage induced during flight . The extent to which constitutive IFN-a expression signifies constitutive antiviral defense in the form of functional IFN-a protein remains unresolved.', 'metadata': {'title': 'Accelerated viral dynamics in bat cell lines, with implications for zoonotic emergence', 'doc_id': 'sample_97_doc_0', 'chunk_id': 'sample_97_doc_0_chunk_0', 'sample_index': 97}, 'faiss_score': 0.9204219579696655, 'rerank_score': 0.9773678183555603}, {'text': '

## 20. Batch Retrieval Evaluation

In [121]:
import pandas as pd

retrieval_results = []

for idx in range(min(10, len(dataset))):

    sample = dataset[idx]

    query = sample.get("question")

    results = retrieve(query, top_k=5)

    generated_response = generate_response(
        query,
        results
    )

    scores = judge_trace_scores(
        query,
        results,
        generated_response
    )

    # ---------------------------------------------
    # Format retrieved docs nicely
    # ---------------------------------------------

    retrieved_docs_text = "\n\n".join([
    f"""
[Rank {rank+1}]
Title: {doc['metadata']['title']}
Doc ID: {doc['metadata']['doc_id']}
Chunk ID: {doc['metadata']['chunk_id']}

FAISS Score: {doc['faiss_score']:.4f}
Rerank Score: {doc['rerank_score']:.4f}

Content:
{doc['text'][:300]}
""".strip()
    for rank, doc in enumerate(results)
])

    # ---------------------------------------------
    # Store everything row-wise
    # ---------------------------------------------

    retrieval_results.append({
        "Query": query,
        "Retrieved Docs": retrieved_docs_text,
        "LLM Response": generated_response,
        "Trace Relevance": f"{round(scores.get('relevance_score', 0), 4)} | GT: {round(sample['relevance_score'], 4)}",
        "Trace Utilization": f"{round(scores.get('utilization_score', 0), 4)} | GT: {round(sample['utilization_score'], 4)}",
        "Trace Completeness": f"{round(scores.get('completeness_score', 0), 4)} | GT: {round(sample['completeness_score'], 4)}",
        "Trace Adherence": f"{scores.get('adherence_score', '')} | GT: {sample['adherence_score']}"
    })

# -------------------------------------------------
# Create dataframe
# -------------------------------------------------

df_results = pd.DataFrame(retrieval_results)

# Optional:
# widen column display
pd.set_option("display.max_colwidth", None)

display(df_results)

,Query,Retrieved Docs,LLM Response,Trace Relevance,Trace Utilization,Trace Completeness,Trace Adherence
0,Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?,"[Rank 1]\nTitle: Accelerated viral dynamics in bat cell lines, with implications for zoonotic emergence\nDoc ID: sample_97_doc_0\nChunk ID: sample_97_doc_0_chunk_0\n\nFAISS Score: 0.9204\nRerank Score: 0.9774\n\nContent:\ntranscriptomic blueprints for this IFN response are expressed constitutively, even in the absence of stimulation by viral RNA or DNA . In non-flying mammals, constitutive IFN expression would likely elicit widespread inflammation and concomitant immunopathology upon viral infection, but bats support\n\n[Rank 2]\nTitle: Advanced Molecular Surveillance of Hepatitis C Virus\nDoc ID: sample_189_doc_0\nChunk ID: sample_189_doc_0_chunk_1\n\nFAISS Score: 0.9180\nRerank Score: 0.9718\n\nContent:\nSeveral studies have suggested that differences in host immune responses determine viral clearance. Indeed, the intensity of the immune response can be dictated, at least in part, by host genetic factors . However, the role of host factors in HCV clearance is not well understood. Different studies h\n\n[Rank 3]\nTitle: Confounding roles for type I interferons during bacterial and viral pathogenesis\nDoc ID: sample_64_doc_3\nChunk ID: sample_64_doc_3_chunk_1\n\nFAISS Score: 0.9265\nRerank Score: 0.9464\n\nContent:\nHowever, the interaction between viral and bacterial infection is not always deleterious. Infection with herpesvirus induces prolonged IFN-γ production that leads to protection against infection with L. monocytogenes and Yersinia pestis . The main distinguishing feature between\n\n[Rank 4]\nTitle: Type I Interferon Receptor Deficiency in Dendritic Cells Facilitates Systemic Murine Norovirus Persistence Despite Enhanced Adaptive Immunity\nDoc ID: sample_0_doc_0\nChunk ID: sample_0_doc_0_chunk_0\n\nFAISS Score: 0.9416\nRerank Score: 0.9381\n\nContent:\nsuccessful treatment for HCV serves to circumvent the viral inhibition of IFN induction. Thus, HCV may be an example of a medically relevant persistent viral infection that persists due, in part, to loss of innate immune function. Persistence of other continuously replicating RNA viruses, such as ch\n\n[Rank 5]\nTitle: Interferon-Induced Ifit2/ISG54 Protects Mice from Lethal VSV Neuropathogenesis\nDoc ID: sample_97_doc_1\nChunk ID: sample_97_doc_1_chunk_1\n\nFAISS Score: 0.9330\nRerank Score: 0.7438\n\nContent:\nIn these mice, although IFN is induced by virus infection, it cannot act on target cells. Similarly, in genetically altered mice that are defective in IFN production due to the absence of specific pathogen-associated pattern recognition receptors, signaling proteins or specific transcription factors","Based on the provided context, the viruses that may not cause prolonged inflammation due to strong induction of antiviral clearance are those that are effectively countered by the host's innate immune response, particularly through the action of type I interferons (IFN). \n\nDocument 1 suggests that bats have unique adaptations to combat inflammation, which may be related to their constitutive IFN expression. Document 3 shows that infection with herpesvirus can induce prolonged IFN-γ production, leading to protection against other infections. Document 4 implies that some viruses, such as HCV, may persist due to ineffective innate immune responses, but it does not directly address the question of which viruses may not cause prolonged inflammation.\n\nDocument 5, however, provides a clue. It states that IFN-induced Ifit2/ISG54 protects mice from lethal VSV neuropathogenesis. This suggests that strong induction of antiviral clearance, mediated by IFN, can prevent prolonged inflammation and viral pathogenesis.\n\nTherefore, based on the provided context, the viruses that may not cause prolonged inflammation due to strong induction of antiviral clearance are likely t

## 21. Save Index and Metadata

In [122]:
# Save FAISS index
faiss.write_index(index, "faiss_index.bin")
print("FAISS index saved to faiss_index.bin")

# Save documents and metadata
with open("documents.json", "w") as f:
    json.dump({
        "documents": documents,
        "metadata": metadata
    }, f, indent=2)
print("Documents and metadata saved to documents.json")

# Save retrieval results
df_results.to_csv(
    "retrieval_results.csv",
    index=False
)
print("Retrieval results saved to retrieval_results.csv")

FAISS index saved to faiss_index.bin
Documents and metadata saved to documents.json
Retrieval results saved to retrieval_results.csv


## 24. Load Index (for future use)

In [ ]:
# To load the index later:
# index = faiss.read_index("faiss_index.bin")
# with open("documents.json", "r") as f:
#     data = json.load(f)
#     documents = data["documents"]
#     metadata = data["metadata"]
# model = SentenceTransformer('BAAI/bge-large-en-v1.5')

print("Index loading code ready!")